# RDDT ATTR v3 - WT01 end-to-end

**Signal:** WT01 = RULE_ATTRWT_ORTHO_BEFORE_CARDIO (temporal T01).
Not an atom / not a bucket - chronology that upgrades ORTHO+CARDIO.

**Pipeline:**
SOURCE tables -> evidence -> atoms -> WT04 ORTHO_CLUSTER -> bucket tiers -> T01 -> GA01 -> AC01 -> WT_RULE_01/02 -> guardrails -> router

Configs: specialty_configs_v3/ atoms / overlays / composites / temporal / combinations / guardrails.


## 0 - Session + import


In [ ]:
import importlib
import sys
from pathlib import Path
import pandas as pd
from snowflake.snowpark.context import get_active_session

HERE = Path.cwd()
V3_CANDIDATES = [HERE, HERE / 'specialty_configs_v3', Path('/tmp/specialty_configs_v3'), Path('/tmp')]
_v3 = next(
    (p for p in V3_CANDIDATES
     if (p / 'pipeline.py').exists() and (p / 'rddt_attr_sql.py').exists() and (p / 'atoms').is_dir()),
    None,
)
if _v3 is None:
    raise FileNotFoundError('Upload / mount specialty_configs_v3/')
if str(_v3) not in sys.path:
    sys.path.insert(0, str(_v3))

import pipeline as wt01
import rddt_attr_sql as rsql
importlib.reload(wt01)
importlib.reload(rsql)
session = get_active_session()
print('config:', _v3)
session


In [ ]:
DATABASE = 'RDDT'
SCHEMA = 'PUBLIC'
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {SCHEMA}').collect()
print(session.get_current_database(), session.get_current_schema())


## 1 - SOURCE_CONFIG (your tables)

Edit physical names only if the client renames columns.


In [ ]:
SOURCE_CONFIG = wt01.default_source_config()
TEXT_COLUMNS = wt01.TEXT_COLUMNS
ICD_CODE_COLUMNS = wt01.ICD_CODE_COLUMNS
PROCEDURE_CODE_COLUMNS = wt01.PROCEDURE_CODE_COLUMNS
SNOMED_CODE_COLUMNS = wt01.SNOMED_CODE_COLUMNS

for logical, source in SOURCE_CONFIG['tables'].items():
    flag = 'on ' if source.get('enabled', True) else 'off'
    print(f"[{flag}] {logical:18s} -> {source['name']}")

rsql.validate_sources(session, SOURCE_CONFIG, raise_on_error=True)
n_pat = rsql.patient_count(session, SOURCE_CONFIG)
print('patients:', f'{n_pat:,}')
rsql.table_qc(session, SOURCE_CONFIG)


## 2 - Load WT01 triggers from atoms JSON

ATTRwt ORTHO/CARDIO overlays + WT04 members + mgus.


In [ ]:
CONFIG_ROOT = wt01.find_v3_config_root()
vocab = wt01.build_wt01_vocab_frames(CONFIG_ROOT)
print('atoms', len(vocab['atom_ids']))
print('codes', len(vocab['codes_df']), '| keywords', len(vocab['keywords_df']), '| patterns', len(vocab['patterns_df']))
for p in wt01.keyword_wildcard_variations('bilateral CTS'):
    print(' ', p)
display(vocab['codes_df'].head(20))
display(vocab['keywords_df'].head(20))


## 3 - Run WT01 end-to-end

Stages 1-3: wide net -> candidates -> ATTR_EVID_*
Stages 4+: atoms -> WT04 -> tiers -> T01 (WT01) -> combinations -> guardrails -> router


In [ ]:
result = wt01.run_full_attrwt_pipeline(session, SOURCE_CONFIG, CONFIG_ROOT)
print('--- stages 1-3 ---')
print({k: v for k, v in result['stages_1_3'].items() if k != 'evidence_inventory'})
display(result['stages_1_3']['evidence_inventory'])
print('--- phenotype layers ---')
print(result['stages_4_plus'])


## 4 - Inspect each layer


In [ ]:
print('--- BUCKET_TIER ATTRwt ---')
display(wt01.preview_bucket_tiers(session, 'ATTRwt', 40))
print('--- WT04 ORTHO_CLUSTER ---')
display(wt01.preview_composites(session, 40))
print('--- T01 / WT01 ---')
display(wt01.preview_t01(session, 40))
print('--- Combinations ---')
display(wt01.preview_combinations(session, 60))
print('--- Guardrails ---')
display(session.sql('SELECT * FROM ATTR_V3_GUARDRAIL_HITS LIMIT 40').to_pandas())
print('--- ROUTER ---')
display(wt01.preview_router(session, 50))


## 5 - Success check

Gate-eligible ORTHO+CARDIO with ORTHO before CARDIO -> T01_SATISFIED=TRUE and WT_RULE_01. Missing dates -> WT_RULE_02. CARDIO-before-ORTHO -> no ATTRwt rule.


In [ ]:
wt01_hits = session.sql('''
  SELECT PATIENT_ID, ATTRWT, ATTRWT_RULE_ID, ATTRWT_ROUTE,
         T01_SATISFIED, T01_YEARS_BETWEEN, ORTHO_FIRST_SEEN, CARDIO_FIRST_SEEN,
         GENERAL_AMYLOID, ATTR_COMMON, GUARDRAIL_ROUTES
  FROM ATTR_V3_ROUTER_OUTPUT
  WHERE ATTRWT_RULE_ID IN ('WT_RULE_01', 'WT_RULE_02')
     OR T01_SATISFIED = TRUE
  ORDER BY T01_YEARS_BETWEEN DESC NULLS LAST
  LIMIT 100
''').to_pandas()
print('WT01-related router rows:', len(wt01_hits))
wt01_hits
